# Comparação das soluções para a aorta

Compara o pipeline normal e o filtro agressivo nas mesmas imagens de treino e validação. A análise mantém separados os rótulos visuais, o status automático dos óstios e as métricas de segmentação.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in [current, *current.parents]
    if (path / "src").exists() and (path / "output").exists()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    get_aorta_visual_review,
    load_aorta_visual_reviews,
    resolve_aorta_review_summary_path,
)
from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
pd.set_option("display.max_columns", 60)

## 1. Runs e classificações

Os IDs e as notas da inspeção ficam centralizados em `config/aorta_visual_reviews.json`.

In [ ]:
REVIEW_CONFIG_PATH = REPO_ROOT / "config/aorta_visual_reviews.json"
REVIEW_CATALOG = load_aorta_visual_reviews(REVIEW_CONFIG_PATH)
SPLITS = ("train", "val")
VARIANTS = ("normal", "aggressive")
DISPLAY_NAMES = {"normal": "Normal", "aggressive": "Agressivo"}
SPLIT_NAMES = {"train": "Treino", "val": "Validação"}

reviews = {
    (split, variant): get_aorta_visual_review(REVIEW_CATALOG, variant, split)
    for split in SPLITS
    for variant in VARIANTS
}
for split in SPLITS:
    for variant in VARIANTS:
        review = reviews[(split, variant)]
        print(
            f"{SPLIT_NAMES[split]} | {DISPLAY_NAMES[variant]}: "
            f"{len(review['aorta_bad_ids'])} aortas ruins e "
            f"{len(review['ostia_bad_ids'])} óstios ruins "
            f"(fonte: {review['ostia_review_source']})"
        )

## 2. Carregamento pareado

In [ ]:
COMPARISON_COLUMNS = [
    "IMG_ID", "artery_dice", "ostia_detection_status",
    "aorta_mask_voxel_count", "aorta_volume_fraction",
    "aorta_voxels_per_segmented_slice", "aorta_circle_count",
    "aorta_circle_filter_accepted", "aorta_level_set_profile_used",
]


def load_solution(split, variant):
    """Carrega uma solução e acrescenta os rótulos da revisão correspondente."""
    review = reviews[(split, variant)]
    summary_path = resolve_aorta_review_summary_path(REPO_ROOT, review, split)
    solution_df = pd.read_csv(summary_path)
    missing = set(COMPARISON_COLUMNS).difference(solution_df.columns)
    if missing:
        raise ValueError(f"Colunas ausentes em {variant}/{split}: {sorted(missing)}")

    solution_df = solution_df[COMPARISON_COLUMNS].copy()
    solution_df["IMG_ID"] = pd.to_numeric(solution_df["IMG_ID"], errors="raise").astype(int)
    expected_ids = review["aorta_good_ids"] | review["aorta_bad_ids"]
    observed_ids = set(solution_df["IMG_ID"])
    if observed_ids != expected_ids:
        raise ValueError(
            f"IDs incompatíveis em {variant}/{split}: "
            f"ausentes={sorted(expected_ids - observed_ids)}; "
            f"não revisados={sorted(observed_ids - expected_ids)}"
        )

    normalized_status = (
        solution_df["ostia_detection_status"].astype(str).str.lower()
        .str.replace("_", " ", regex=False)
    )
    solution_df["csv_ostia_success"] = normalized_status.isin(
        {"both correct", "both tolerable", "both ostia correct", "both ostia tolerable"}
    )
    solution_df["visual_aorta_good"] = solution_df["IMG_ID"].isin(review["aorta_good_ids"])
    solution_df["reviewed_ostia_success"] = solution_df["IMG_ID"].isin(review["ostia_good_ids"])
    solution_df["ostia_review_source"] = review["ostia_review_source"]
    solution_df["review_note"] = solution_df["IMG_ID"].map(review["notes"]).fillna("")
    solution_df["variant"] = variant
    solution_df["split"] = split
    return solution_df


solutions = {
    (split, variant): load_solution(split, variant)
    for split in SPLITS
    for variant in VARIANTS
}
paired = {}
for split in SPLITS:
    normal_df = solutions[(split, "normal")]
    aggressive_df = solutions[(split, "aggressive")]
    if set(normal_df["IMG_ID"]) != set(aggressive_df["IMG_ID"]):
        raise ValueError(f"As soluções de {split} não contêm os mesmos IMG_IDs.")
    paired_df = normal_df.merge(
        aggressive_df, on="IMG_ID", suffixes=("_normal", "_aggressive")
    )
    paired_df["dice_delta"] = (
        paired_df["artery_dice_aggressive"] - paired_df["artery_dice_normal"]
    )
    paired_df["aorta_volume_delta"] = (
        paired_df["aorta_volume_fraction_aggressive"]
        - paired_df["aorta_volume_fraction_normal"]
    )
    paired[split] = paired_df
    print(f"{SPLIT_NAMES[split]}: {len(paired_df)} imagens pareadas")

## 3. Resultado geral

As linhas permanecem separadas por subconjunto e solução.

In [ ]:
def build_overview_row(split, variant):
    """Resume qualidade visual, óstios, Dice e volume de uma solução."""
    frame = solutions[(split, variant)]
    return {
        "subconjunto": SPLIT_NAMES[split],
        "solução": DISPLAY_NAMES[variant],
        "imagens": len(frame),
        "aortas_boas": int(frame["visual_aorta_good"].sum()),
        "taxa_aorta_boa": frame["visual_aorta_good"].mean(),
        "sucesso_óstios_revisado": frame["reviewed_ostia_success"].mean(),
        "fonte_óstios": frame["ostia_review_source"].iat[0],
        "sucesso_óstios_csv": frame["csv_ostia_success"].mean(),
        "dice_médio": frame["artery_dice"].mean(),
        "dice_mediano": frame["artery_dice"].median(),
        "volume_aorta_médio_%": frame["aorta_volume_fraction"].mean() * 100,
    }


overview_df = pd.DataFrame(
    build_overview_row(split, variant)
    for split in SPLITS
    for variant in VARIANTS
)
display(overview_df.round(4))

## 4. Mudanças na avaliação visual

In [ ]:
def build_transition_table(split):
    """Lista correções, regressões e falhas comuns dentro de uma coorte."""
    normal = reviews[(split, "normal")]
    aggressive = reviews[(split, "aggressive")]
    normal_bad_aorta = normal["aorta_bad_ids"]
    aggressive_bad_aorta = aggressive["aorta_bad_ids"]
    normal_bad_ostia = normal["ostia_bad_ids"]
    aggressive_bad_ostia = aggressive["ostia_bad_ids"]
    table = pd.DataFrame(
        {
            "desfecho": [
                "aorta corrigida", "aorta piorou", "aorta ruim nas duas",
                "óstios corrigidos", "óstios pioraram", "óstios ruins nas duas",
            ],
            "IMG_IDs": [
                sorted(normal_bad_aorta - aggressive_bad_aorta),
                sorted(aggressive_bad_aorta - normal_bad_aorta),
                sorted(normal_bad_aorta & aggressive_bad_aorta),
                sorted(normal_bad_ostia - aggressive_bad_ostia),
                sorted(aggressive_bad_ostia - normal_bad_ostia),
                sorted(normal_bad_ostia & aggressive_bad_ostia),
            ],
        }
    )
    table.insert(0, "subconjunto", SPLIT_NAMES[split])
    table["quantidade"] = table["IMG_IDs"].str.len()
    return table[["subconjunto", "desfecho", "quantidade", "IMG_IDs"]]


transition_df = pd.concat(
    [build_transition_table(split) for split in SPLITS], ignore_index=True
)
display(transition_df)

## 5. Comparação gráfica

In [ ]:
def plot_solution_comparison(split):
    """Compara falhas, Dice pareado e volume da aorta em uma coorte."""
    paired_df = paired[split]
    normal_review = reviews[(split, "normal")]
    aggressive_review = reviews[(split, "aggressive")]

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    x = np.arange(2)
    axes[0].bar(
        x - 0.18,
        [len(normal_review["aorta_bad_ids"]), len(aggressive_review["aorta_bad_ids"])],
        width=0.36, label="Aorta ruim", color="#d1495b",
    )
    axes[0].bar(
        x + 0.18,
        [len(normal_review["ostia_bad_ids"]), len(aggressive_review["ostia_bad_ids"])],
        width=0.36, label="Óstios ruins", color="#e9c46a",
    )
    axes[0].set_xticks(x, ["Normal", "Agressivo"])
    axes[0].set_ylabel("Quantidade de imagens")
    axes[0].set_title("Falhas revisadas")
    axes[0].legend()

    ordered = paired_df.sort_values("artery_dice_normal").reset_index(drop=True)
    axes[1].plot(ordered.index, ordered["artery_dice_normal"], label="Normal", linewidth=1.5)
    axes[1].plot(ordered.index, ordered["artery_dice_aggressive"], label="Agressivo", linewidth=1.5)
    axes[1].set(xlabel="Exames ordenados pelo baseline", ylabel="Dice", title="Dice por exame", ylim=(0, 1))
    axes[1].legend()

    colors = np.where(paired_df["visual_aorta_good_aggressive"], "#2a9d8f", "#d1495b")
    axes[2].scatter(
        paired_df["aorta_volume_fraction_normal"] * 100,
        paired_df["aorta_volume_fraction_aggressive"] * 100,
        c=colors, s=48, alpha=0.85, edgecolor="white",
    )
    limit = max(axes[2].get_xlim()[1], axes[2].get_ylim()[1])
    axes[2].plot([0, limit], [0, limit], linestyle="--", color="#555555", linewidth=1)
    axes[2].set(
        xlabel="Volume da aorta normal (%)",
        ylabel="Volume da aorta agressivo (%)",
        title="Mudança no volume da máscara",
    )
    for ax in axes:
        ax.grid(alpha=0.2)
    fig.suptitle(SPLIT_NAMES[split], fontsize=14)
    fig.tight_layout()
    plt.show()


plot_solution_comparison("train")

In [ ]:
plot_solution_comparison("val")

## 6. Casos alterados e observações

In [ ]:
def build_changed_cases(split):
    """Retorna apenas exames cuja avaliação ou Dice mudou de forma relevante."""
    paired_df = paired[split].copy()
    changed = paired_df.loc[
        paired_df["visual_aorta_good_normal"].ne(paired_df["visual_aorta_good_aggressive"])
        | paired_df["reviewed_ostia_success_normal"].ne(
            paired_df["reviewed_ostia_success_aggressive"]
        )
        | paired_df["dice_delta"].abs().ge(0.02)
    ].copy()
    changed["review_note"] = changed["review_note_aggressive"].where(
        changed["review_note_aggressive"].ne(""), changed["review_note_normal"]
    )
    columns = [
        "IMG_ID", "visual_aorta_good_normal", "visual_aorta_good_aggressive",
        "reviewed_ostia_success_normal", "reviewed_ostia_success_aggressive",
        "artery_dice_normal", "artery_dice_aggressive", "dice_delta", "review_note",
    ]
    return changed[columns].sort_values("dice_delta", ascending=False)


print("Treino")
display(build_changed_cases("train").round(4))

In [ ]:
print("Validação")
display(build_changed_cases("val").round(4))

## 7. Interpretação

A comparação deve considerar simultaneamente qualidade visual da aorta, desfecho dos óstios e Dice. Uma máscara visualmente melhor não garante automaticamente melhor localização dos óstios. No treino normal, a coluna revisada dos óstios usa o status do próprio summary porque não houve uma lista visual independente; essa origem permanece explícita na tabela geral.